In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
def corr_map_plot(df):
    # Compute the correlation matrix
    corr_matrix = df.corr()

    # Plot the correlation map
    plt.figure(figsize = (10, 8))
    sns.heatmap(corr_matrix, annot = True, cmap = 'coolwarm', vmin = -1, vmax = 1)
    plt.title("Correlation Map")
    plt.savefig('CorrelationMap.png')
    
    return corr_matrix

In [2]:
def find_most_corr(corr_matrix, df):
    corr_matrix_out = corr_matrix.iloc[:-1,-1]

    pos_feature_name = corr_matrix_out.idxmax()
    neg_feature_name = corr_matrix_out.idxmin()

    max_index = int(df.columns.get_loc(pos_feature_name))
    min_index = int(df.columns.get_loc(neg_feature_name))
                
    return max_index, min_index, pos_feature_name, neg_feature_name

In [3]:
def scatterplot_features(df, index, feature_name):
    plt.figure(figsize = (10, 5))
    plt.scatter(df.iloc[:, index], df.iloc[:, -1], edgecolor = 'k')
    plt.title('Scatter plot of ' + feature_name + ' vs Rings')
    plt.xlabel(feature_name)
    plt.ylabel('Rings')
    plt.savefig(feature_name + 'ScatterPlotFeature.png')

In [ ]:
def histogram_plot(df, index, feature_name):
    plt.figure(figsize = (10, 5))
    plt.hist(df.iloc[:, index], bins = 50, color = 'skyblue', edgecolor = 'black')
    plt.title('Histogram of ' + feature_name)
    plt.xlabel(feature_name)
    plt.ylabel('Count')
    plt.grid(True, which = 'both', linestyle = '--', linewidth = 0.5)
    plt.savefig(feature_name + 'Histogram.png')

In [ ]:
def data_processing(df):
    # Clean the data
    # Replace 'M', 'F', 'I' with 0, 1, 2 respectively
    df[0] = df[0].replace({'M': 0, 'F': 1, 'I': 2})
    # Set the header to each column
    df.columns = ["Sex", "Length", "Diameter", "Height", "Whole_weight", "Shucked_weight", "Viscera_weight", "Shell_weight", "Rings"]
    
    corr_matrix = corr_map_plot(df)
    max_index, min_index, pos_feature_name, neg_feature_name = find_most_corr(corr_matrix, df)
    
    scatterplot_features(df, max_index, pos_feature_name)
    scatterplot_features(df, min_index, neg_feature_name)
    
    histogram_plot(df, max_index, pos_feature_name)
    histogram_plot(df, min_index, neg_feature_name)

    return df, max_index, min_index

In [ ]:
def random_split(abalone, experiment_number, train_percent):
    seed = experiment_number
    train, test = train_test_split(abalone, test_size = 1 - train_percent, random_state = seed)
    return train, test

In [ ]:
def linear_reg_model(Xtrain, ytrain, Xtest):
    # Initialize the linear regression model
    linear_regression = LinearRegression()
    # Train the model
    linear_regression.fit(Xtrain, ytrain)
    # Predictions
    ypred = linear_regression.predict(Xtest)
    return ypred

In [ ]:
def scatterplot_prediction(i, ytrue, ypred):
    # Scatter plot of true vs predicted values
    plt.figure(figsize = (10, 6))
    plt.scatter(ytrue, ypred, color = 'blue')
    plt.plot([min(ytrue), max(ytrue)], [min(ytrue), max(ytrue)], color = 'red')
    plt.xlabel('True Values')
    plt.ylabel('Predicted Values')
    plt.title('True vs Predicted Ring-Age')
    plt.savefig(str(i) + 'ScatterPlotPrediction.png')

In [ ]:
def fit_normalizing(normalize, Xtrain, Xtest, ytrain):
    # Initialize MinMaxScaler
    scaler = MinMaxScaler()

    # Normalize Xtrain and Xtest separately to prevent data leakage
    Xtrain_norm = scaler.fit_transform(Xtrain)
    Xtest_norm = scaler.transform(Xtest)

    # Linear Regression Model
    ypred_norm = linear_reg_model(Xtrain_norm, ytrain, Xtest_norm)

    return ypred_norm

In [ ]:
def cal_RMSE_R2(ytrue, ypred):
    # Calculate RMSE and R-squared
    rmse = np.sqrt(mean_squared_error(ytrue, ypred))
    r2 = r2_score(ytrue, ypred)

    print("Root Mean Squared Error (RMSE):", rmse)
    print("R-squared Score:", r2)
    
    return rmse

In [ ]:
def get_in_out(train, test):
    # Features and target variable
    Xtrain = train[:,:-1]
    ytrain = train[:,-1]
    Xtest = test[:,:-1]
    ytest = test[:,-1]
    return Xtrain, ytrain, Xtest, ytest

In [ ]:
def neural_network_model(n, lr, Xtrain):
    NNmodel = Sequential()
    NNmodel.add(Dense(n, activation = 'relu', input_shape = (Xtrain.shape[1],)))
    NNmodel.add(Dense(1))
    optimizer = SGD(learning_rate = lr)
    NNmodel.compile(loss = 'mean_squared_error', optimizer = optimizer)
    return NNmodel

In [ ]:
def neural_network(neurons, learning_rates, Xtrain, ytrain, Xtest, ytest, i):
    best_val_loss = np.inf
    best_params = {}
    for n in neurons:
        for lr in learning_rates:
            print(f"Training model {i} with {n} neurons and learning rate {lr}...")
            NNmodel = neural_network_model(n, lr, Xtrain)
            history = NNmodel.fit(Xtrain, ytrain, epochs = 100, batch_size = 32, 
                                  validation_data=(Xtest, ytest), verbose = 0)
            
            final_val_loss = history.history['val_loss'][-1]
            
            if final_val_loss < best_val_loss:
                best_val_loss = final_val_loss
                best_params = {'neurons': n, 'learning_rate': lr}
    print(f"Best model has {best_params['neurons']} neurons and learning rate {best_params['learning_rate']} with validation loss {best_val_loss}")
    return n, lr, best_val_loss

In [ ]:
def linear_model(data, experiment_number, train_percent, normalize, i):
    train, test = random_split(data, experiment_number, train_percent)
    Xtrain, ytrain, Xtest, ytest = get_in_out(train, test)
    if normalize == True:
        ypred = fit_normalizing(normalize, Xtrain, Xtest, ytrain)
    else:
        ypred = linear_reg_model(Xtrain, ytrain, Xtest)
    scatterplot_prediction(i, ytest, ypred)
    rmse = cal_RMSE_R2(ytest, ypred)
    return Xtrain, ytrain, Xtest, ytest, rmse

In [ ]:
def modelling(abalone, max_index, min_index):

    experiment_number = 30
    train_percent = 0.6

    # Experiment with different neurons and learning rates
    neurons = [3, 4, 5, 6, 7, 8]
    learning_rates = [0.0001, 0.001, 0.01, 0.1]
    
    # Model 1: All features without normalizing
    normalize1 = False
    Xtrain1, ytrain1, Xtest1, ytest1, rmse1 = linear_model(abalone, experiment_number, train_percent, normalize1, 1)    

    # Model 2: All features with normalizing
    normalize2 = True
    Xtrain2, ytrain2, Xtest2, ytest2, rmse2 = linear_model(abalone, experiment_number, train_percent, normalize2, 2)

    # Model 3: Two selected features without normalizing
    abalone_sele = abalone[:,(max_index, min_index, -1)]
    print(abalone_sele)
    normalize3 = False
    Xtrain3, ytrain3, Xtest3, ytest3, rmse3 = linear_model(abalone_sele, experiment_number, train_percent, normalize3, 3)

    if rmse1 < rmse2:
        if rmse1 < rmse3:
            neural_network(neurons, learning_rates, Xtrain1, ytrain1, Xtest1, ytest1, 1)
        else:
            neural_network(neurons, learning_rates, Xtrain3, ytrain3, Xtest3, ytest3, 3)
    else:
        if rmse2 < rmse3:
            neural_network(neurons, learning_rates, Xtrain2, ytrain2, Xtest2, ytest2, 2)
        else:
            neural_network(neurons, learning_rates, Xtrain3, ytrain3, Xtest3, ytest3, 3)
            

In [ ]:
if __name__ == "__main__":    
    # Read the data
    df = pd.read_csv('abalone_data.csv', sep = ',', header = None)
    df, max_index, min_index = data_processing(df)
    abalone = df.values
    modelling(abalone, max_index, min_index)